In [1]:
!pip install "numpy==1.26.4" "scikit-learn==1.6.1" "joblib==1.5.3" "pandas==2.2.2"


In [5]:
!pip install \
  "langchain==0.3.7" \
  "langchain-core==0.3.18" \
  "langchain-openai==0.2.3" \
  "langchain-community==0.3.7" \
  "langchain-pinecone==0.2.0" \
  "langchain-huggingface==0.1.2" \
  "langgraph==0.2.19" \
  "sentence-transformers==5.2.0" \
  "pinecone-client==5.0.1" \
  "gradio==6.2.0" \
  "pinecone-plugin-interface"\
  "pypdf"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 7.3 MB/s eta 0:00:00


In [3]:
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from pinecone import Pinecone, ServerlessSpec
from google.colab import userdata
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.tools import tool
import os


os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")
pc = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

hf_api_key = userdata.get("HF_TOKEN")
if hf_api_key:
  os.environ["HF_TOKEN"] = hf_api_key

endpoint_llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-4B-Instruct-2507",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto"
)
llm = ChatHuggingFace(llm=endpoint_llm)

/tmp/ipython-input-562448611.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

                    provider was transferred to model_kwargs.
                    Please make sure that provider is what you intended.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Base de datos vectorial para el mantenimiento de una casa

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document



loader = PyPDFLoader("guia para el buen uso de la vivienda_para web.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

guide_chunks = text_splitter.split_documents(documents=docs)

if "guia-mantenimiento" not in pc.list_indexes().names():
    pc.create_index(
        name="guia-mantenimiento",
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("Índice guia-mantenimiento creado.")
else:
    print("Índice guia-mantenimiento ya existe.")



mantenimiento_db = PineconeVectorStore.from_documents(
    documents=guide_chunks,
    embedding=embeddings,
    index_name="guia-mantenimiento"
)

Índice guia-mantenimiento ya existe.


Base de datos vectorial para los 7 errores al vender una casa

In [8]:
loader = PyPDFLoader("erroresAlVenderTuPropiedad.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

guide_chunks = text_splitter.split_documents(documents=docs)

if "errores-venta-propiedad" not in pc.list_indexes().names():
    pc.create_index(
        name="errores-venta-propiedad",
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("Índice errores-venta-propiedad creado.")
else:
    print("Índice errores-venta-propiedad ya existe.")

errores_db = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name="errores-venta-propiedad"
)

Índice errores-venta-propiedad ya existe.


Definimos las tools

In [9]:
from __future__ import annotations

from typing import Any, Dict, List, Optional
import pandas as pd


def _toSiNo(value: Any) -> str:
    """
    Normalize booleans / strings to 'Si' / 'No'.
    Accepts: 'Si'/'No', 'sí'/'no', True/False, 1/0, None.
    """
    if value is None:
        return "No"

    if isinstance(value, bool):
        return "Si" if value else "No"

    if isinstance(value, (int, float)):
        return "Si" if value != 0 else "No"

    if isinstance(value, str):
        v = value.strip().lower()
        if v in ("si", "sí", "s", "true", "t", "1", "yes", "y"):
            return "Si"
        if v in ("no", "false", "f", "0", "n"):
            return "No"
    return "No"


def _toOptionalInt(value: Any) -> Optional[int]:
    if value is None or value == "":
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def _toOptionalFloat(value: Any) -> Optional[float]:
    if value is None or value == "":
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def buildVentaInputRow(
    inputData: Dict[str, Any],
    expectedColumns: List[str],
) -> pd.DataFrame:
    """
    Builds a single-row dataframe aligned to the columns used in training (raw feature columns).
    Any missing column is created with default None/'No'.
    Any extra field in inputData is ignored unless present in expectedColumns.

    expectedColumns should be the EXACT training columns, e.g.:
      expectedColumns = list(XTrain.columns)
    """
    # Defaults base
    baseRecord: Dict[str, Any] = {
        # Core
        "tipoInmueble": inputData.get("tipoInmueble"),
        "condicion": inputData.get("condicion"),
        "barrio": inputData.get("barrio"),
        "departamento": inputData.get("departamento"),

        # Amenities (Si/No)
        "terraza": _toSiNo(inputData.get("terraza")),
        "patio": _toSiNo(inputData.get("patio")),
        "toilette": _toSiNo(inputData.get("toilette")),
        "aircond": _toSiNo(inputData.get("aircond")),
        "calefacc": _toSiNo(inputData.get("calefacc")),
        "jardin": _toSiNo(inputData.get("jardin")),
        "piscina": _toSiNo(inputData.get("piscina")),
        "garage": _toSiNo(inputData.get("garage")),
        "kitchenette": _toSiNo(inputData.get("kitchenette")),
        "losaRad": _toSiNo(inputData.get("losaRad")),
        "parrillero": _toSiNo(inputData.get("parrillero")),
        "salaReuniones": _toSiNo(inputData.get("salaReuniones")),
        "seguridad": _toSiNo(inputData.get("seguridad")),
        "amoblado": _toSiNo(inputData.get("amoblado")),
        "comedor": _toSiNo(inputData.get("comedor")),

        # Numeric
        "banos": _toOptionalInt(inputData.get("banos")),
        "apPpiso": _toOptionalInt(inputData.get("apPpiso")),
        "dormitorios": _toOptionalInt(inputData.get("dormitorios")),
        "expensas": _toOptionalFloat(inputData.get("expensas")),
        "supConstru": _toOptionalFloat(inputData.get("supConstru")),
        "antiguedad": _toOptionalInt(inputData.get("antiguedad")),
        "ambientes": _toOptionalInt(inputData.get("ambientes")),
        "ascensores": _toOptionalInt(inputData.get("ascensores")),
        "supTot": _toOptionalFloat(inputData.get("supTot")),

        # Categorical
        "tipoEdif": inputData.get("tipoEdif"),
        "estado": inputData.get("estado"),
        "orientacion": inputData.get("orientacion"),
    }

    # Alinear exactamente a expectedColumns
    aligned: Dict[str, Any] = {}
    for col in expectedColumns:
        if col in baseRecord:
            aligned[col] = baseRecord[col]
        else:
            # si la col existía en training y no la tenemos acá, la dejamos en None
            aligned[col] = None

    dfRow = pd.DataFrame([aligned], columns=expectedColumns)
    return dfRow


In [10]:
import joblib

bundle = joblib.load("modelo_inmobiliario.pkl")
modeloCasas = bundle["model"]
expectedColumns = bundle["expectedColumns"]


In [11]:
@tool
def recomendacion_buen_uso_vivienda(query : str) -> str:
  """
    Busca información sobre el buen uso de la vivienda y sobre definiciones relacionadas a la vivienda.
    Usa esta herramienta para recomendarle al usuario buenas prácticas para mantener la vivienda
    y cuando pregunte por definiciones específicas relacionadas a la vivienda.
  """
  try:
    docs = mantenimiento_db.similarity_search(query, k=3)
    if not docs:
      return "No se encontró información relevante en los papers cargados."

    results = []
    for i, doc in enumerate(docs, 1):
      source = doc.metadata.get("source", "Desconocido")
      results.append(f"[Documento {i} - {source}]\n{doc.page_content}")

    return "\n\n---\n\n".join(results)
  except Exception as e:
    return f"Error al buscar en papers: {str(e)}"


@tool
def errores_al_vender_propiedad(query : str) -> str:
  """
    Busca información sobre los errores más comunes al vender una propiedad.
  """
  try:
    docs = errores_db.similarity_search(query, k=3)
    if not docs:
      return "No se encontró información relevante en los papers cargados."

    results = []
    for i, doc in enumerate(docs, 1):
      source = doc.metadata.get("source", "Desconocido")
      results.append(f"[Documento {i} - {source}]\n{doc.page_content}")

    return "\n\n---\n\n".join(results)
  except Exception as e:
    return f"Error al buscar en papers: {str(e)}"


@tool
def predecir_precio_venta_propiedad(inputData: Dict[str, Any]) -> float:
  """
    Predice el precio de venta de una propiedad en base a sus características.

    Usa el modelo entrenado (modeloCasas) y las columnas de entrenamiento
    (expectedColumns). El parámetro `inputData` debe ser un diccionario con
    campos como: tipoInmueble, condicion, barrio, departamento, garage,
    banos, dormitorios, supTot, estado, orientacion, etc.
  """
  try:
    dfRow = buildVentaInputRow(
        inputData=inputData,
        expectedColumns=expectedColumns
    )

    yPred = modeloCasas.predict(dfRow)[0]

    return float(yPred)

  except Exception as e:
    raise RuntimeError(f"Error al predecir el precio: {e}")


tools = [recomendacion_buen_uso_vivienda, errores_al_vender_propiedad, predecir_precio_venta_propiedad]

In [12]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

system_instructions = """
Eres un asistente conversacional inteligente con acceso a bases de conocimiento especializadas.

REGLAS IMPORTANTES:
1. SMALL-TALK: Para saludos, chistes, conversación casual, responde directamente SIN usar herramientas.

2. CUANDO USAR HERRAMIENTAS:
   - recomendacion_buen_uso_vivienda: Cuando el usuario pregunte sobre cómo mantener de la mejor forma su
   vivienda o sobre definiciones especificas del rubro.
   - errores_al_vender_propiedad: Cuando el usuario pregunte que no deberia hacer al vender una propiedad,
   tambien puedes tener en cuenta los errores para darle recomendaciones al usuario sobre como vender.
   - predecir_precio_venta_propiedad: Cuando el usuario quiera una estimacion del precio de venta de su propiedad.
   Debe brindarte la siguiente informacion, no toda es obligatoria: tipo de inmueble (Apartamentos, Casas), condicion (used, new), Barrio,
   Departamento, terraza (Si, No), patio (Si, No), toilette (Si, No), aircond (Si, No), calefacc (Si, No),
   jardin (Si, No), piscina (Si, No), garage (Si, No), kitchenette (Si, No), losa_rad (Si, No), parrillero (Si, No),
   sala_reuniones (Si, No), seguridad (Si, No), amoblado (Si,No),
   banos (cantidad), ap_ppiso (cantidad), dormitorios (cantidad), tipo_edif, estado ('Muy bueno', 'Bueno', 'Excelente', 'A refaccionar', 'Regular'),
   expensas (numero), sup_constru (numero), orientacion ('Contrafrente', 'Este', 'Frente', 'Interno', 'Lateral', 'Norte', 'Oeste', 'Sur'),
   antiguedad (numero), ambientes (numero), ascensores (numero), sup_tot (numero), comedor (Si, No). Debes pasarlo en el siguiente formato:
      inputData = {
          "tipoInmueble": "Apartamentos",
          "condicion": "used",
          "barrio": "Pocitos",
          "departamento": "Montevideo",
          "garage": "Si",
          "banos": 2,
          "dormitorios": 3,
          "supTot": 90,
          "estado": "Muy bueno",
          "orientacion": "Frente"
      }
    IMPORTANTE: Puedes recibir una consulta en texto plano, tu te debes encargar de pasarlo al formato correcto.
    NOTA: No juzgues el modelo, solo di lo que te dio.

3. CONTROL DE ALUCINACIONES:
   - SI NO TIENES INFORMACIÓN: Admite honestamente que no tienes la información necesaria.
   - NO INVENTES: Nunca inventes datos, nombres, fechas o información que no esté en las fuentes.
   - USA SOLO LO QUE ENCUENTRAS: Si una herramienta no encuentra información relevante,
     di claramente "No encontré información sobre [tema] en las bases de conocimiento disponibles."
   - CITA FUENTES: Cuando uses información de herramientas, menciona que proviene de los documentos cargados.

4. MEMORIA: Mantén contexto de la conversación para responder preguntas de seguimiento.

5. RESPUESTAS: Sé claro, conciso y útil. Si no estás seguro, es mejor admitirlo que inventar.
"""

# Se bindean las tools al LLM.
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: MessagesState):
    """Nodo del chatbot que mantiene memoria y permite tool_calls."""
    messages = [SystemMessage(content=system_instructions)]

    messages.extend(state["messages"])

    result = llm_with_tools.invoke(messages)
    ai_msg = AIMessage(content=result.content, tool_calls=result.tool_calls)

    return {"messages": state["messages"] + [ai_msg]}

# ===== Construcción del grafo =====
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)


graph = graph_builder.compile()

In [13]:
conversation_state = {"messages": []}

def chat_with_agent(user_input: str, reset: bool = False):
    """
    Función para interactuar con el agentic RAG con memoria persistente.

    Args:
        user_input: Pregunta o mensaje del usuario
        reset: Si True, reinicia la conversación (limpia la memoria)
    """
    global conversation_state

    if reset:
        conversation_state = {"messages": []}
        print("Conversación reiniciada.\n")

    print(f"Usuario: {user_input}")
    print("\n" + "="*60)
    print("Procesando...")
    print("="*60 + "\n")

    conversation_state["messages"].append(HumanMessage(content=user_input))
    response = graph.invoke(conversation_state)

    conversation_state = response

    last_message = response["messages"][-1]

    print("Asistente:", last_message.content)
    print("\n" + "="*60)

    return last_message.content


In [14]:
import gradio as gr

# --- CHAT WRAPPER ---
def gradio_chat(message, history):
    """
    Función wrapper para Gradio ChatInterface.
    Esta función adapta chat_with_agent para trabajar con la interfaz de Gradio.
    """
    global conversation_state

    if not conversation_state.get("messages"):
        conversation_state = {"messages": []}
    conversation_state["messages"].append(HumanMessage(content=message))
    response = graph.invoke(conversation_state)
    conversation_state = response
    last_message = response["messages"][-1]

    return last_message.content


def estimar_precio(tipoInmueble, barrio, condicion, dormitorios, banos, supTot, garage, estado, orientacion):
    try:
        inputData = {
            "tipoInmueble": tipoInmueble,
            "condicion": condicion,
            "barrio": barrio,
            "departamento": "Montevideo",
            "dormitorios": dormitorios,
            "banos": banos,
            "supTot": supTot,
            "garage": garage,
            "estado": estado,
            "orientacion": orientacion,
        }
        dfRow = buildVentaInputRow(inputData, expectedColumns)
        pred = modeloCasas.predict(dfRow)[0]
        return f"USD {pred:,.0f}"
    except Exception as e:
        return f"Error al estimar precio: {e}"


theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="slate",
)


with gr.Blocks(title="Asistente Inmobiliario Inteligente", theme=theme) as demo:

    gr.Markdown(
        """
        # 🏠 Asistente Inmobiliario Inteligente

        **Entrenado con datos reales del mercado de Montevideo (2018)**
        - Responde dudas sobre **buen uso y mantenimiento de la vivienda**
        - Te muestra **errores comunes al vender una propiedad**
        - Puede **estimar el precio de venta** según características básicas
        """
    )

    with gr.Tab("💬 Consultar por chat"):
        gr.ChatInterface(
            fn=gradio_chat,
            examples=[
                "Quiero vender mi apartamento en Pocitos, ¿qué errores debería evitar?",
                "¿Cómo mantengo en buen estado una casa antigua?",
                "Tengo una casa en La Teja, 2 dorm y 1 baño, ¿qué precio puede tener?",
                "¿Qué información necesitas para estimar el precio de mi vivienda?",
            ],
        )

    with gr.Tab("📊 Estimar precio de propiedad"):
        gr.Markdown("### Completa los datos de tu propiedad (Montevideo, 2018)")

        tipoInmueble = gr.Dropdown(["Apartamentos", "Casas"], label="Tipo de inmueble")
        barrio = gr.Textbox(label="Barrio", placeholder="Pocitos, La Teja, Carrasco, etc.")
        condicion = gr.Dropdown(["new", "used"], label="Condición")
        dormitorios = gr.Slider(0, 6, step=1, value=2, label="Dormitorios")
        banos = gr.Slider(0, 4, step=1, value=1, label="Baños")
        supTot = gr.Number(label="Superficie total (m²)", value=70)
        garage = gr.Dropdown(["Si", "No"], label="Garage")
        estado = gr.Dropdown(["A reciclar", "Regular", "Bueno", "Muy bueno", "Excelente"], label="Estado")
        orientacion = gr.Dropdown(["Frente", "Contrafrente", "Lateral"], label="Orientación")

        btn = gr.Button("Calcular precio estimado")
        out = gr.Textbox(label="Precio estimado (USD aprox.)")

        btn.click(
            estimar_precio,
            inputs=[tipoInmueble, barrio, condicion, dormitorios, banos, supTot, garage, estado, orientacion],
            outputs=out,
        )

demo.launch(share=True)


/tmp/ipython-input-2683928651.py:48: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Asistente Inmobiliario Inteligente", theme=theme) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://007312b84baf41109e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
